# 사용자를 기억하는 agent 만들기

대부분의 agent는 매번 대화를 처음부터 시작합니다. 고객이 쇼핑 assistant에 사이즈, 예산, 피하는 소재를 알려 줘도 다음에 돌아오면 agent는 모든 것을 잊었습니다. 고객은 되풀이해야 하며 경험은 개인화되기보다 획일적으로 느껴집니다.

이를 해결하기 위해 Claude Managed Agents에 Memory를 막 도입했습니다. agent가 고객마다 받는 공유 notebook으로 생각하세요. Claude는 세션 도중 관련 내용을 적고, 같은 고객이 다음에 돌아와도 메모는 그대로 있습니다. 간단한 API 호출로 설정할 수 있습니다.

이 guide에서는 소매 브랜드의 예제 쇼핑 assistant를 만듭니다. agent는 첫 방문에서 고객의 선호를 배우고 사용자별 memory store에 저장하며, 다음 방문에서는 다시 알려 주지 않아도 자동으로 기억해 냅니다.

## 만들 내용

- 고객 한 명의 쇼핑 선호를 보관하는 **memory store**
- 해당 store를 확인하고 update하도록 설정된 **쇼핑 agent**
- 방문 사이에 memory가 이어지는 것을 보여 주는 **두 개의 별도 세션**
- 자체 애플리케이션에서 memory를 **살펴보고 미리 넣는** 패턴

## memory 작동 방식

memory store를 세션에 연결하면 agent environment 안의 `/mnt/memory/{store-name}`에 디렉터리로 나타나며, Claude는 표준 파일 tool로 그곳의 파일을 읽고 씁니다. 애플리케이션은 REST API를 통해 같은 파일에 완전한 읽기와 쓰기 권한을 가지므로 알려진 사실로 store를 미리 채우거나, agent가 쓴 내용을 audit하거나, 자체 시스템으로 모든 것을 export할 수 있습니다.

> **Beta 기능.** Memory store는 Claude Managed Agents public beta의 일부입니다. 정식 출시 전에 API가 바뀔 수 있습니다. Python SDK는 `client.beta` 아래 모든 method에 필요한 `anthropic-beta` header를 자동으로 추가합니다.


## 사전 요구 사항

- Claude Managed Agents beta에 접근할 수 있는 Anthropic API key. `ANTHROPIC_API_KEY` environment variable로 설정하세요.
- Python 3.11 이상.
- Anthropic Python SDK. Memory store method에는 최신 release가 필요합니다.

```bash
uv add anthropic
# 또는: pip install -U anthropic
```


## client 설정하기


In [ ]:
%%capture
%pip install -q "anthropic>=0.91.0"

In [1]:
import os

from anthropic import Anthropic
from utilities import wait_for_idle_status

MODEL = os.environ.get("COOKBOOK_MODEL", "claude-sonnet-4-6")

client = Anthropic()

### 대화 turn용 helper

Managed agent 세션은 event 기반입니다. user message를 보낸 뒤 세션이 idle이 될 때까지 event를 streaming합니다. guide의 나머지를 읽기 쉽게 유지하려면 이 루프를 agent reply와 `/mnt/memory/` 아래 접근한 파일을 출력하는 helper로 감쌉니다.


In [2]:
def run_turn(session_id: str, user_text: str) -> str:
    """user message 하나를 보내고 agent가 idle이 될 때까지 응답을 streaming합니다.

    프로그램에서 사용하려는 caller에게 agent의 전체 text reply를 반환합니다.
    """
    print(f"\n[user]  {user_text}")
    reply_parts: list[str] = []

    with client.beta.sessions.events.stream(session_id) as stream:
        client.beta.sessions.events.send(
            session_id,
            events=[
                {
                    "type": "user.message",
                    "content": [{"type": "text", "text": user_text}],
                }
            ],
        )
        for event in stream:
            if event.type == "agent.message":
                for block in event.content:
                    if block.type == "text":
                        reply_parts.append(block.text)
                        print(f"[agent] {block.text}")

            elif event.type == "agent.tool_use":
                # memory mount의 읽기와 쓰기를 표시하여 agent가 memory를 확인하고
                # update하는 것을 볼 수 있게 합니다.
                inp = event.input or {}
                target = inp.get("file_path") or inp.get("command", "")
                if "/mnt/memory/" in str(target):
                    print(f"  [memory] {event.name}: {target}")

            elif event.type == "session.status_idle":
                # 예상하지 못한 stop_reason(예: requires_action) 때문에
                # 루프가 멈추지 않도록 모든 idle 사유에서 종료합니다.
                break

            elif event.type == "session.status_terminated":
                break

    wait_for_idle_status(client, session_id)
    return "".join(reply_parts)

## 1단계: memory store 생성하기

memory store는 workspace 범위의 text 파일용 이름이 있는 container입니다. production deployment에서는 보통 최종 사용자별로 store 하나를 생성하고 자체 database에 사용자 ID와 store ID의 mapping을 보관합니다.

여기서 설정한 `description`은 store가 연결될 때마다 agent의 system prompt에 표시되므로 Claude에 store 용도를 알려 주는 데 사용하세요.


In [3]:
store = client.beta.memory_stores.create(
    name="Shopper Preferences",
    description=(
        "Personal shopping preferences for a single customer: "
        "sizes, style, budget, favorite brands, and materials to avoid."
    ),
)

print(store.id)  # memstore_01...

memstore_01NKkumXZXY8mEhoRA3xhBvN


## 2단계: 쇼핑 agent 정의하기

모든 managed agent 세션에는 **agent**(model, system prompt, tool)와 **environment**(agent가 실행되는 container)가 필요합니다. 한 번 생성하여 여러 고객과 세션에서 재사용할 수 있습니다.

memory를 읽고 쓰는 파일 tool이 포함된 built-in `agent_toolset`을 제공하세요. `agent_toolset_20260401` 타입 문자열은 model alias가 아니라 toolset의 API 식별자이므로 새 model이 출시되어도 update할 필요가 없습니다.


In [4]:
environment = client.beta.environments.create(
    name="shopping-demo",
    config={"type": "cloud", "networking": {"type": "limited"}},
)

agent = client.beta.agents.create(
    name="Personal Shopper",
    # 이 agent가 어느 cookbook에서 왔는지 Anthropic에 알립니다. 제거해도 안전합니다.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-remember-user-preferences"},
    model=MODEL,
    system=(
        "You are a personal shopping assistant for a retail brand. "
        "Help the customer find products that match their taste and budget, "
        "and remember what you learn about them for future visits."
    ),
    tools=[
        {
            "type": "agent_toolset_20260401",
            "default_config": {
                "enabled": True,
                "permission_policy": {"type": "always_allow"},
            },
        }
    ],
)

print(agent.id)  # agent_01...

agent_011CaMbJoHMUyR5TjeFDtMG9


## 3단계: agent가 선호를 배우는 첫 방문

이제 세션을 시작하고 `resources` 배열로 memory store를 연결합니다. `instructions` 필드는 연결별 안내로, Claude에 이 특정 세션에서 이 특정 store를 어떻게 사용할지 알려 줍니다.

고객의 첫 방문이므로 store는 비어 있습니다. 출력의 `[memory]` 줄을 보세요. agent가 store를 확인하여 아직 아무것도 없음을 알아낸 다음 배운 내용을 담은 새 파일을 씁니다.


In [5]:
memory_resource = {
    "type": "memory_store",
    "memory_store_id": store.id,
    "access": "read_write",
    "instructions": (
        "This customer's personal preferences: sizes, style, budget, and "
        "materials to avoid. Check it at the start of every conversation "
        "and update it whenever you learn something new."
    ),
}

session_one = client.beta.sessions.create(
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    environment_id=environment.id,
    resources=[memory_resource],
)

# 응답은 agent environment 내부의 store mount 위치를 알려 줍니다.
for resource in session_one.resources:
    if resource.type == "memory_store":
        print(f"Mounted at {resource.mount_path}")

Mounted at /mnt/memory/shopper-preferences


In [6]:
run_turn(
    session_one.id,
    "Hi! I'm looking for a new jacket. A few things about me: I wear a size "
    "medium, I only buy vegan leather (no animal leather please), my budget "
    "is usually under $200, and I love earth tones. What would you suggest?",
)

[user]  Hi! I'm looking for a new jacket. A few things about me: I wear a size medium, I only buy vegan leather (no animal leather please), my budget is usually under $200, and I love earth tones. What would you suggest?
[agent] Let me check if I have any info about you on file, and save your preferences right away!
  [memory] bash: cat /mnt/memory/shopper-preferences 2>/dev/null || echo "No file found"
  [memory] write: /mnt/memory/shopper-preferences
  [memory] write: /mnt/memory/shopper-preferences/preferences.md
[agent] I've saved your preferences for future visits! Now, here are some great **vegan leather jacket picks in earth tones under $200** that I'd suggest:

---

### 🌿 Top Picks for You

#### 1. **Olive Moto Jacket** — ~$110–$150
A classic biker silhouette in a deep **olive green** faux leather. Look for styles with zip-detail sleeves and a fitted waist. Great for casual or edgy looks.
- *Why you'll love it:* Earthy tone, very on-trend, versatile for layering.

#### 2. **Cam

## 4단계: agent가 저장한 내용 살펴보기

agent가 mount에 쓰는 모든 것은 애플리케이션이 API로 읽고, 편집하고, 삭제할 수 있는 일반 memory 문서입니다. 응답에 파일 내용을 포함하도록 `view="full"`로 store의 내용을 나열하세요.

이렇게 하면 제품에 "우리가 당신에 대해 아는 내용" 페이지를 만들거나, 자체 database에 memory를 동기화하거나, 사람이 agent의 잘못을 교정하게 할 수 있습니다.


In [7]:
page = client.beta.memory_stores.memories.list(
    store.id,
    view="full",
)

for memory in page.data:
    if memory.type == "memory":
        print(f"=== {memory.path} ===")
        print(memory.content)
        print()

=== /preferences.md ===
# Shopper Preferences

## Sizes
- Tops/Jackets: Medium

## Style
- Loves earth tones (browns, tans, olive, terracotta, camel, rust, etc.)
- Interested in jackets

## Budget
- Usually under $200

## Materials
- VEGAN LEATHER ONLY — absolutely no animal leather
- (No other material restrictions noted yet)

## Favorite Brands
- None noted yet

## Other Notes
- First visit; preferences collected 2026-04-23


고객이 언급한 사이즈, 예산, 소재, 색상 선호가 주제별로 정리된 파일(보통 `/preferences.md` 같은 파일)이 보여야 합니다. Claude가 파일명과 구조를 직접 선택했습니다.


## 5단계: agent가 스스로 기억해 내는 재방문

이 순간이 중요합니다. **완전히 새로운 세션**을 만들고 **같은 memory store**를 연결합니다. 고객은 선호를 되풀이하지 않지만 agent는 memory에서 읽어 그에 맞게 추천합니다.


In [8]:
session_two = client.beta.sessions.create(
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    environment_id=environment.id,
    resources=[memory_resource],  # 같은 store, 새 세션
)

run_turn(
    session_two.id,
    "Hey, I'm back! I need a bag for work. Any recommendations?",
)

[user]  Hey, I'm back! I need a bag for work. Any recommendations?
[agent] Welcome back! Let me check your preferences before making any recommendations!
  [memory] read: /mnt/memory/shopper-preferences
  [memory] bash: ls /mnt/memory/shopper-preferences/
  [memory] read: /mnt/memory/shopper-preferences/preferences.md
[agent] Great news — I already have your preferences on file! Here's what I kept in mind for you:

- 🎨 **Style:** Earth tones (browns, tans, olive, camel, etc.)
- 💰 **Budget:** Under $200
- 🌱 **Materials:** Vegan leather only — no animal leather

With that in mind, here are some work bag recommendations that would suit you perfectly:

---

### 🛍️ Top Picks for a Work Bag

1. **Structured Tote in Camel Vegan Leather** — A polished, professional look in a warm earth tone. Fits a laptop + daily essentials. Great desk-to-commute bag.

2. **Olive Canvas & Vegan Leather Trim Laptop Bag** — A sleek crossbody/shoulder bag with a relaxed but put-together vibe. The canvas keeps it 

출력에서 agent가 답하기 전에 `/mnt/memory/shopper-preferences/`를 읽고, message에 전혀 언급하지 않았는데도 vegan leather 소재, earth tone 색상, $200 미만의 가방을 추천하는 점에 주목하세요. 첫 세션의 선호가 이어졌습니다.


## 더 나아가기: production 패턴

### 기존 데이터로 store 미리 채우기

계정 profile이나 구매 이력에서 이미 고객에 대해 알고 있는 내용이 있다면 첫 세션 전에 store에 써서 agent가 정보를 알고 시작하게 할 수 있습니다. 실제 애플리케이션에서는 세션을 생성하기 전에 이 초기 데이터 입력 단계를 실행합니다. 여기서는 위의 주요 학습 후 기억 흐름에 집중하도록 demo 뒤에만 배치했습니다.

> **참고:** notebook 마지막 정리 셀에서 `store`를 삭제하므로 이 셀은 그 전에 실행하세요.


In [9]:
# 선택 사항: 정리 셀 전에 실행하세요.
seeded = client.beta.memory_stores.memories.create(
    store.id,
    path="/purchase-history.md",
    content=(
        "## Recent purchases\n"
        "- Canvas tote, olive, $89 (Jan 2026)\n"
        "- Wool beanie, rust, $34 (Dec 2025)\n"
    ),
)
print(f"Seeded {seeded.path}")

Seeded /purchase-history.md


### 고객별 store와 공유 store 결합하기

세션은 각각 고유한 접근 수준을 가진 memory store를 최대 여덟 개 연결할 수 있습니다. 일반적인 패턴은 고객별 읽기·쓰기 store 하나에 모든 세션이 공유하는 브랜드 전체 지식의 읽기 전용 store를 더하는 것입니다.

```python
catalog = client.beta.memory_stores.create(
    name="Product Catalog Notes",
    description="Current promotions, sizing guidance, and stock notes.",
)

session = client.beta.sessions.create(
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    environment_id=environment.id,
    resources=[
        {
            "type": "memory_store",
            "memory_store_id": store.id,
            "access": "read_write",
            "instructions": "This customer's personal preferences.",
        },
        {
            "type": "memory_store",
            "memory_store_id": catalog.id,
            "access": "read_only",
            "instructions": "Brand-wide product guidance. Consult before recommending items.",
        },
    ],
)

# 완료하면 catalog store를 정리하세요:
# client.beta.memory_stores.delete(catalog.id)
```

### audit와 교정

memory store에 쓰는 모든 내용은 작성한 세션과 함께 immutable version으로 기록됩니다. `client.beta.memory_stores.memory_versions.list(...)`로 이력을 검토하고, `client.beta.memory_stores.memories.update(...)`로 agent가 잘못 작성한 파일을 교정하세요. 전체 인터페이스는 [Memory API reference](https://docs.anthropic.com/en/docs/managed-agents/memory)를 참고하세요.


## 정리하기

이 guide를 따라 생성한 resource를 삭제하세요.


In [10]:
wait_for_idle_status(client, session_one.id)
wait_for_idle_status(client, session_two.id)

client.beta.sessions.archive(session_one.id)
client.beta.sessions.archive(session_two.id)
client.beta.memory_stores.delete(store.id)
client.beta.agents.archive(agent.id)
client.beta.environments.archive(environment.id)

## 요약

다음 방법으로 방문 사이에 고객을 기억하는 쇼핑 agent를 만들었습니다.

1. 고객용 memory store 생성하기
2. `resources`로 각 세션에 연결하기
3. 새 선호를 배우면서 Claude가 `/mnt/memory/` 아래 파일을 읽고 쓰게 하기
4. Memory API로 자체 애플리케이션에서 해당 파일 살펴보기

이제 자체 사용자 ID를 memory store ID에 mapping하고, 기존 고객 데이터로 store를 미리 채우고, 브랜드 전체 지식을 위한 공유 읽기 전용 store를 추가할 수 있습니다.

### 이 시리즈의 다른 notebook

- [`CMA_iterate_fix_failing_tests.ipynb`](CMA_iterate_fix_failing_tests.ipynb) — 시작점 notebook입니다. 실패하는 test suite의 실행-관찰-수정 루프를 통해 agent, environment, session, 파일 mount, streaming event 루프를 소개합니다.
- [`CMA_operate_in_production.ipynb`](CMA_operate_in_production.ipynb) — production 설정 설명입니다. vault에 저장된 MCP credential, 장시간 연결 없이 HITL을 수행하는 `session.status_idled` webhook, resource lifecycle의 CRUD 동작을 다룹니다.

### 자세히 알아보기

- [Claude Managed Agents 개요](https://docs.anthropic.com/en/docs/managed-agents/overview)
- [Memory stores API reference](https://docs.anthropic.com/en/docs/managed-agents/memory)
- [Session resource](https://docs.anthropic.com/en/docs/managed-agents/memory#attach-a-memory-store-to-a-session)
